## Faz a classificação dos cargos, locais e salários
Ao final, salva arquivo para posterior análise da habilidades ("skills")

In [1]:
import pandas as pd
from unidecode import unidecode

### Parte 0: carrega dados

In [2]:
# Carrega os dados brutos
df = pd.read_csv('data/scp_01.csv')
print(df.shape)

(15869, 5)


In [3]:
# Carrega critérios
df_palavras_chave = pd.read_excel('data/criterios.xlsx', sheet_name='palavras_chave')
df_palavras_chave = df_palavras_chave.sort_values(by='Prioridade') 

df_tipo_cargo = pd.read_excel('data/criterios.xlsx', sheet_name='tipo_cargo')
df_tipo_cargo = df_tipo_cargo.sort_values(by='Prioridade')

df_nivel_cargo = pd.read_excel('data/criterios.xlsx', sheet_name='nivel_cargo')
df_nivel_cargo = df_nivel_cargo.sort_values(by='Prioridade')

df_cidades = pd.read_excel('data/criterios.xlsx', sheet_name='cidades')
df_salario = pd.read_excel('data/criterios.xlsx', sheet_name='salario')

### Parte 2: Classificando

In [4]:
# Faz o match dos critérios para área, tipo e nível do cargo
def match_rows(dff, text):
    if text == '' or pd.isna(text):
        return 'n/e' # not exist
    res = 'n/f' # not found
    rows = dff[dff["Cargo"].apply(lambda k: f' {k} ' in f' {text} ')] 
    if rows.shape[0] > 0:
        for _, row in rows.iterrows(): 
            for column in dff.columns[2:]:
                if row[column] == 1:
                    res = column
                    break
    return res

In [5]:
import re

# ── Índice normalizado (acento + caixa) para matching fuzzy ───────────────
def _norm(t):
    return unidecode(str(t)).lower().strip()

_cidades_idx = {_norm(loc): i for i, loc in df_cidades['Local'].items()}

# ── Padrões para detectar modalidade remoto/híbrido na descrição ──────────
# Conservadores: evitam falsos positivos como "metodologias híbridas" ou
# "remote team" sem indicar que A VAGA é remota.
_PAT_REMOTO = re.compile(
    r'(?:'
    r'100\s*%\s*remoto'
    r'|modelo\s+(?:de\s+trabalho\s+)?remoto'
    r'|regime\s+(?:de\s+trabalho\s+)?remoto'
    r'|formato\s+remoto'
    r'|vaga\s+remota'
    r'|fully\s+remote'
    r'|trabalhar\s+remotamente'
    r'|local\s+do\s+trabalho\s*[:\-]?\s*remoto'
    r')',
    re.I,
)

_PAT_HIBRIDO = re.compile(
    r'(?:'
    r'modelo\s+(?:de\s+trabalho\s+)?h[íi]brid'
    r'|regime\s+h[íi]brid'
    r'|trabalho\s+h[íi]brid'
    r'|vaga\s+h[íi]brid'
    r'|atua[çc][ãa]o\s+h[íi]brid'
    r'|formato\s+h[íi]brid'
    r'|local\s+do\s+trabalho\s*[:\-]?\s*h[íi]brid'
    r')',
    re.I,
)

def match_local(text, skills_text=''):
    if not isinstance(text, str) or text.strip() == '':
        return ['n/e', 'n/e', 'n/e']

    local_cidade = 'n/f'
    local_estado = 'n/f'
    local_tipo   = 'n/f'

    # 1. Match exato na planilha
    rows = df_cidades[df_cidades['Local'] == text]

    # 2. Match normalizado (ignora acentos e caixa)
    if len(rows) == 0:
        norm = _norm(text)
        if norm in _cidades_idx:
            rows = df_cidades.iloc[[_cidades_idx[norm]]]

    # 3. Extração de estado quando formato é "Cidade, Estado"
    #    útil para cidades desconhecidas que estejam no padrão "Franca, São Paulo"
    if len(rows) == 0 and ',' in text:
        estado_str = text.split(',')[-1].strip()
        rows_uf = df_cidades[df_cidades['Local'] == estado_str]
        if len(rows_uf) == 0:
            norm_uf = _norm(estado_str)
            if norm_uf in _cidades_idx:
                rows_uf = df_cidades.iloc[[_cidades_idx[norm_uf]]]
        if len(rows_uf) > 0:
            local_estado = rows_uf.iloc[0]['Local Estado']
            local_tipo   = 'nacional'

    if len(rows) > 0:
        row = rows.iloc[0]
        local_cidade = row['Local Cidade']
        local_estado = row['Local Estado']
        local_tipo   = row['Local Tipo']

    # 4. Detecção de modalidade pela descrição da vaga
    #    só atua quando a planilha não definiu remoto/internacional
    if local_tipo not in ('remoto', 'internacional') and isinstance(skills_text, str) and skills_text:
        if _PAT_REMOTO.search(skills_text):
            local_tipo = 'remoto'
        elif _PAT_HIBRIDO.search(skills_text):
            local_tipo = 'hibrido'

    return [local_cidade, local_estado, local_tipo]

In [6]:
def match_salario(text):
    if text == '' or pd.isna(text):
        return 'n/e'
    salario_faixa = 'n/f'
    rows = df_salario[df_salario['Salario'] == text]
    if rows.shape[0] > 0:
        row = rows.iloc[0]
        salario_faixa = row['faixa']
    return salario_faixa

In [7]:
for index, row in df.iterrows():
    df.at[index, 'Area']  = match_rows(df_palavras_chave, row['Cargo'])
    df.at[index, 'Tipo']  = match_rows(df_tipo_cargo, row['Cargo'])
    df.at[index, 'Nivel'] = match_rows(df_nivel_cargo, row['Cargo'])

    loc = match_local(row['Local'], row['Skills'])
    df.at[index, 'Local Cidade'] = loc[0]
    df.at[index, 'Local Estado'] = loc[1]
    df.at[index, 'Local Tipo']   = loc[2]

    df.at[index, 'Salario2'] = match_salario(row['Salario'])

### Parte 3: Ajusta e salva os dados

In [8]:
# Salva esse arquivo para conferir se as classificações estão adequadas
# Se não estiverem, é preciso ajustar os dados nas planilhas do arquivo "criterios.xlslx"
df[['Cargo', 'Area', 'Tipo', 'Nivel']].dropna().sort_values(by=['Cargo']).to_excel('data/scp_02_chk.xlsx', index=False)

In [9]:
# Carrega a blacklist do Excel
df_blacklist = pd.read_excel('data/criterios.xlsx', sheet_name='blacklist')

lista_proibida = [unidecode(str(termo)).lower().strip() for termo in df_blacklist['Termo'].tolist()]

padrao_regex = '|'.join(lista_proibida)

df = df[~df['Cargo'].str.contains(padrao_regex, case=False, na=False)]

print(f"Salvando arquivo limpo com {df.shape[0]} linhas...")

df[['Cargo', 'Area', 'Tipo', 'Nivel']].dropna().sort_values(by=['Cargo']).to_excel('data/scp_02_clean.xlsx', index=False)

Salvando arquivo limpo com 15521 linhas...


In [10]:
# Remove e renomeia colunas
df.drop(columns=['Empresa', 'Local', 'Salario'], inplace=True)
df.rename(columns={'Salario2': 'Salario'}, inplace=True)
df = df[df["Local Tipo"] != 'internacional'] # remove vagas internacionais

In [11]:
# Salva arquivo
df.to_csv('data/scp_02.csv', index=False)